# Parte III — Redes e Fluxos · o código dos capítulos 16 a 22

**Este caderno não contém o algoritmo.** Ele busca o código publicado do handbook e chama as
mesmas funções que o `pytest` do repositório verifica — não existe segunda cópia para envelhecer
([ADR 0016](https://github.com/GHDaru/operationalresearchaibook/blob/main/adr/0016-cadernos-colab-sem-deriva.md)).

Rode as células em ordem. No Colab, `Ctrl+F9` roda tudo.

> **Ele roda igual na sua máquina.** O Colab é conveniência, não dependência.


In [ ]:
# 1. Traz o código publicado. Sem magias do IPython: Python puro roda igual
#    no Colab e no seu terminal — e é o que o teste consegue executar.
import subprocess, sys
from pathlib import Path

URL = "https://github.com/GHDaru/operationalresearchaibook"
RAIZ = Path("operationalresearchaibook")

if not RAIZ.exists():
    subprocess.run(["git", "clone", "--depth", "1", URL, str(RAIZ)], check=True)

ETAPA = RAIZ / "po-zero" / "parte-III-redes"
sys.path.insert(0, str(ETAPA.resolve()))
print("código em:", ETAPA.resolve())

## Capítulo 17 — o método que se contradiz

Dijkstra é rápido porque **fecha** um nó em definitivo. A hipótese que autoriza isso é peso não
negativo — e ela não está no código, está na cabeça de quem escolheu o método.


In [ ]:
# 2. A malha honesta primeiro: quando a hipótese vale, os dois métodos concordam.
from redes import (COM_CICLO_NEGATIVO, COM_PESO_NEGATIVO, MALHA,
                   bellman_ford, caminho, dijkstra)

d = dijkstra(MALHA, "deposito")
b = bellman_ford(MALHA, "deposito")
print("dijkstra e bellman-ford concordam:", d["distancias"] == b["distancias"])
print("caminho ate o cliente:", " -> ".join(caminho(d, "cliente")),
      "custa", d["distancias"]["cliente"])

### O palpite — erre antes de ver o certo

Agora a mesma coisa numa rede com **um peso negativo** (um crédito recebido ao usar o trecho).

**Antes de rodar:** você espera que Dijkstra devolva o número errado, o caminho errado, ou os dois?

Escreva a sua resposta. A próxima célula mostra o que de fato acontece — e é nenhuma das três.


In [ ]:
# 3. O contraexemplo. Olhe as DUAS linhas de saída juntas.
dn = dijkstra(COM_PESO_NEGATIVO, "A")
bn = bellman_ford(COM_PESO_NEGATIVO, "A")

trilha = caminho(dn, "D")
peso = {(u, v): c for u, v, c in COM_PESO_NEGATIVO}
custo_do_caminho = sum(peso[(a, b)] for a, b in zip(trilha, trilha[1:]))

print("dijkstra diz: distancia =", dn["distancias"]["D"])
print("dijkstra diz: caminho   =", " -> ".join(trilha), "  que custa", custo_do_caminho)
print()
print("Nao e o numero errado NEM o caminho errado: e os dois juntos, e eles NAO FECHAM.")
print("A saida contradiz a si mesma, e nenhum aviso e emitido.")
print()
print("bellman-ford diz:", bn["distancias"]["D"], "-- e detecta ciclo negativo:",
      bellman_ford(COM_CICLO_NEGATIVO, "A")["ciclo_negativo"])

## Capítulos 18 e 19 — o mesmo gesto, e o gargalo que não é aresta

In [ ]:
# 4. Guloso: otimo na arvore, 14,3% pior no roteiro. Mesma instancia.
from fractions import Fraction as F
from redes import (CIDADES, REDE, fluxo_maximo, kruskal, mst_por_enumeracao,
                   tsp_exato, tsp_guloso)

k, enu = kruskal(CIDADES), mst_por_enumeracao(CIDADES)
print(f"arvore por Kruskal: {k['custo']}  ·  por enumeracao de TODAS as arvores: {enu['custo']}")
print("  batem:", k["custo"] == enu["custo"], "-- dois caminhos, mesmo numero")

g, e = tsp_guloso(CIDADES, "a"), tsp_exato(CIDADES, "a")
print(f"roteiro guloso: {g['custo']}  ·  roteiro otimo: {e['custo']}"
      f"  ·  perda: {float(F(g['custo'])/F(e['custo'])-1):.1%}")

In [ ]:
# 5. Fluxo maximo = corte minimo, com o corte exibido.
f = fluxo_maximo(REDE, "fabrica", "mercado")
print("fluxo maximo:", f["fluxo"], " ·  capacidade do corte:", f["corte"]["capacidade"],
      " ·  batem:", f["bate"])
print("o corte:", [(u, v) for u, v, _ in f["corte"]["arestas"]])
print()
print("Repare: as tres arestas NAO estao no mesmo nivel da rede.")
print("Gargalo e conjunto que separa, nao etapa -- e investir fora dele nao muda nada.")

## Capítulo 20 — a integralidade que vem de graça, e como ela some

In [ ]:
# 6. Programacao Linear CONTINUA, sem nenhuma variavel inteira declarada.
from redes import (CUSTO, DEMANDA, OFERTA, transporte,
                   transporte_com_estrutura_quebrada)

t = transporte(OFERTA, DEMANDA, CUSTO)
print(f"custo {t['custo']:g}  ·  todos inteiros: {t['todos_inteiros']}")
print("plano:", {r: v for r, v in t["plano"].items() if v})
print()

q = transporte_com_estrutura_quebrada()
print("agora com UMA restricao transversal (espaco de patio):")
print(f"custo {q['custo']:.4f}  ·  todos inteiros: {q['todos_inteiros']}")
print("fracionarios:", {r: round(v, 4) for r, v in q["fracionarios"].items()})
print()
print("`Optimal` nos dois casos. Nada avisou -- a garantia era da FORMA das restricoes.")

## Capítulo 22 — o viés do PERT, e por que ele é menor do que parece

A fórmula do PERT publica **21 dias**. Simulado, o projeto leva **24,48**.

**Antes de rodar:** desses ~3,5 dias de diferença, quanto é culpa do método?

O palpite comum é "tudo". A célula seguinte separa as duas causas **nas mesmas amostras** — e o
número honesto é bem menor.


In [ ]:
# 7. Duas causas, um numero -- e a separacao que custa a manchete.
from redes import (AMOSTRAS_PERT, FAIXAS, PROJETO, SEMENTE, caminho_critico,
                   pert_pela_formula, pert_por_simulacao)

c = caminho_critico(PROJETO)
print(f"CPM: duracao {c['duracao']}  ·  criticas {c['criticas']}  ·  folga do frontend: "
      f"{c['folga']['frontend']}")
print()

pf = pert_pela_formula(FAIXAS, PROJETO)
ps = pert_por_simulacao(FAIXAS, PROJETO, pf["criticas"], AMOSTRAS_PERT, SEMENTE)
print(f"a formula publica:            {pf['duracao_esperada']} dias")
print(f"o projeto leva, em media:     {ps['projeto']['media']:.2f}")
print(f"so o caminho declarado:       {ps['so_o_caminho_declarado']['media']:.2f}")
print(f"MERGE BIAS (mesmas amostras): {ps['merge_bias']}   <-- o palpite comum era ~3,5")
print(f"estoura a estimativa em:      {ps['prob_de_estourar_a_estimativa_do_pert']:.1%}")
print()
print("O resto do desvio vem da distribuicao amostrada nao ter a media que a formula supoe.")
print("Isso e escolha de modelagem, nao defeito do PERT.")

## Mexa aqui

O viés cresce com o número de **ramos paralelos**. Troque `MEUS_RAMOS` abaixo e rode de novo.

Comece por **1**: sem paralelismo, a duração do projeto *é* a do caminho declarado, e o viés tem
de dar **exatamente zero**. Esse zero é o **controle** do experimento — é ele que prova que a
separação está correta.

Depois tente 2, 3, 5 e 8.


In [ ]:
# 8. Sua vez. Mude o numero e rode.
MEUS_RAMOS = 1      # <<< mexa aqui

from redes import rede_com_ramos

tarefas, faixas, ramos = rede_com_ramos(MEUS_RAMOS)
declarado = ["especificar", ramos[0], "integrar"]
s = pert_por_simulacao(faixas, tarefas, declarado, 8000, SEMENTE)

print(f"{MEUS_RAMOS} ramo(s) paralelo(s):")
print(f"  merge bias:            {s['merge_bias']}")
print(f"  estoura a estimativa:  {s['prob_de_estourar_a_estimativa_do_pert']:.1%}")
if MEUS_RAMOS == 1:
    print("  <-- o controle. Tem de dar 0.0, e da.")
print()
print("Repare na segunda linha mesmo com UM ramo: a estimativa ja estoura na maioria")
print("das amostras. Essa parte nao e vies -- e media nao ser promessa.")

## O que o livro não mostra

A célula seguinte imprime **o código-fonte** da função que separa as duas causas. O capítulo
explica o resultado; aqui você lê a implementação — inclusive o comentário que registra, dentro da
própria função, por que a primeira versão dela estava errada.


In [ ]:
# 9. O algoritmo, lido em vez de descrito.
import inspect, redes
print(inspect.getsource(redes.pert_por_simulacao))